In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
file_path_d = r"C:\Users\DME_Lab\DME Dropbox\장성우\Business Process Mining"

In [3]:
# 전체 이벤트 로그
event_log_pm = pd.read_csv(
    os.path.join(file_path_d, 'data', 'G01_eventlog.tsv'),
    sep='\t',
    index_col=0
)

C:\Users\DME_Lab\AppData\Local\Temp\ipykernel_43540\442932569.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  event_log_pm = pd.read_csv(


In [4]:
# 출원 메타데이터
application = pd.read_csv(r"C:\Users\DME_Lab\Documents\수업\BPM\application_data.csv")

C:\Users\DME_Lab\AppData\Local\Temp\ipykernel_43540\1545909521.py:2: DtypeWarning: Columns (0,3,4,6,12,13,14,16,17,18,21) have mixed types. Specify dtype option on import or set low_memory=False.
  application = pd.read_csv(r"C:\Users\DME_Lab\Documents\수업\BPM\application_data.csv")


In [5]:
event_log_pm['recorded_date'] = pd.to_datetime(event_log_pm['recorded_date'], format='mixed', errors='coerce')

# 케이스별 시작일 ~ 종료일
case_span = event_log_pm.groupby('application_number').agg(
    start_date=('recorded_date', 'min'),
    end_date=('recorded_date', 'max')
).reset_index()

# application_number 형식 통일
case_span['application_number'] = case_span['application_number'].astype(str).str.zfill(8)
application['application_number'] = application['application_number'].astype(str).str.zfill(8)

# examiner, art_unit 붙이기
case_span = case_span.merge(
    application[['application_number', 'examiner_full_name', 'examiner_art_unit']],
    on='application_number',
    how='left'
)

print(case_span.shape)
print(case_span.head())

(14071511, 5)
  application_number start_date   end_date  examiner_full_name  \
0           02045760 2001-09-19 2001-09-19                 NaN   
1           02048602 2001-09-19 2001-09-19                 NaN   
2           02107244 1982-03-12 2016-03-08  GREGORY, BERNARR E   
3           02122612 1992-11-06 2001-09-19                 NaN   
4           02122769 2001-09-19 2003-11-10                 NaN   

  examiner_art_unit  
0               NaN  
1               NaN  
2              3662  
3               NaN  
4               NaN  


In [6]:
print(case_span[['examiner_full_name', 'examiner_art_unit']].isna().sum())
print(f"\n비율:")
print(case_span[['examiner_full_name', 'examiner_art_unit']].isna().mean().round(3))

examiner_full_name    2339422
examiner_art_unit     1813404
dtype: int64

비율:
examiner_full_name    0.166
examiner_art_unit     0.129
dtype: float64


In [7]:
# examiner workload용 (examiner 있는 것만)
case_span_exam = case_span.dropna(subset=['examiner_full_name']).copy()

# art unit backlog용 (art unit 있는 것만, examiner 없어도 됨)
case_span_art = case_span.dropna(subset=['examiner_art_unit']).copy()

In [12]:
case_span_art

,application_number,start_date,end_date,examiner_full_name,examiner_art_unit,start_ym,end_ym
2,02107244,1982-03-12,2016-03-08,"GREGORY, BERNARR E",3662,1982-03,2016-03
12,02565523,1985-08-05,1992-07-21,"HUNT, BROOKS H",2204,1985-08,1992-07
13,02568368,1944-12-15,2016-03-08,"GREGORY, BERNARR E",3662,1944-12,2016-03
16,02591067,1979-11-19,2003-09-11,NaN,2202,1979-11,2003-09
17,02602618,1981-11-06,1990-09-12,"BENTLEY, STEPHEN",2201,1981-11,1990-09
...,...,...,...,...,...,...,...
14071506,PCT/ZA20/50056,2020-10-02,2021-08-30,"COPENHEAVER, BLAINE R",1700,2020-10,2021-08
14071507,PCT/ZA20/50057,2020-10-07,2021-08-20,"YOUNG, LEE W",OITP,2020-10,2021-08
14071508,PCT/ZA21/50007,2021-02-16,2021-09-17,"YOUNG, LEE W",OITP,2021-02,2021-09
14071509,PCT/ZA21/50038,2021-06-17,2022-07-19,"SUBRAMANIAN, NARAYANSWAMY",3695,2021-06,2022-07


In [8]:
# examiner workload
case_span_exam['start_ym'] = case_span_exam['start_date'].dt.to_period('M')
case_span_exam['end_ym']   = case_span_exam['end_date'].dt.to_period('M')

start_cnt = case_span_exam.groupby(['examiner_full_name', 'start_ym']).size().rename('delta')
end_cnt   = case_span_exam.groupby(['examiner_full_name', 'end_ym']).size().rename('delta') * -1

delta = pd.concat([start_cnt, end_cnt]).reset_index()
delta.columns = ['examiner_full_name', 'year_month', 'delta']
delta = delta.groupby(['examiner_full_name', 'year_month'])['delta'].sum().reset_index()
delta = delta.sort_values(['examiner_full_name', 'year_month'])
delta['examiner_workload'] = delta.groupby('examiner_full_name')['delta'].cumsum()

print(delta.shape)

(2808171, 4)


In [9]:
# art unit backlog
case_span_art['start_ym'] = case_span_art['start_date'].dt.to_period('M')
case_span_art['end_ym']   = case_span_art['end_date'].dt.to_period('M')

start_cnt_au = case_span_art.groupby(['examiner_art_unit', 'start_ym']).size().rename('delta')
end_cnt_au   = case_span_art.groupby(['examiner_art_unit', 'end_ym']).size().rename('delta') * -1

delta_au = pd.concat([start_cnt_au, end_cnt_au]).reset_index()
delta_au.columns = ['examiner_art_unit', 'year_month', 'delta']
delta_au = delta_au.groupby(['examiner_art_unit', 'year_month'])['delta'].sum().reset_index()
delta_au = delta_au.sort_values(['examiner_art_unit', 'year_month'])
delta_au['artunit_backlog'] = delta_au.groupby('examiner_art_unit')['delta'].cumsum()

print(delta_au.shape)

(407275, 4)


In [10]:
examiner_workload = delta[['examiner_full_name', 'year_month', 'examiner_workload']].copy()
artunit_backlog   = delta_au[['examiner_art_unit', 'year_month', 'artunit_backlog']].copy()

examiner_workload['year_month'] = examiner_workload['year_month'].astype(str)
artunit_backlog['year_month']   = artunit_backlog['year_month'].astype(str)
examiner_workload['examiner_full_name'] = examiner_workload['examiner_full_name'].astype(str).str.strip()
artunit_backlog['examiner_art_unit']    = artunit_backlog['examiner_art_unit'].astype(str).str.strip()

examiner_workload.to_csv(os.path.join(file_path_d, 'data', 'G03_examiner_workload.tsv'), sep='\t', index=False)
artunit_backlog.to_csv(os.path.join(file_path_d, 'data', 'G04_artunit_backlog.tsv'), sep='\t', index=False)

In [13]:
artunit_backlog

,examiner_art_unit,year_month,artunit_backlog
0,0,1989-11,1
1,0,1999-11,3
2,0,2001-09,2
3,0,2001-11,1
4,0,2008-10,0
...,...,...,...
407270,STIC,2023-05,0
407271,WMB,2010-05,1
407272,WMB,2010-10,2
407273,WMB,2022-05,1
